# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saketh0607/flyrank-intership-ml-workflow/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I prioritize existing content pages for refresh review when they have meaningful search visibility, are becoming stale, have an opportunity to improve their search position, or have relatively low content depth.

The baseline uses a transparent deterministic score rather than a fitted machine-learning model. The score combines four signals:

Visibility score — 40%: higher recent impressions indicate greater potential impact.\
Freshness risk score — 30%: older pages receive higher refresh priority.\
Position opportunity score — 25%: pages with better search positions and meaningful visibility have an opportunity to protect or improve their performance.\
Depth gap score — 5%: pages with lower word count relative to the dataset receive a small additional priority.

The final score is between 0 and 1. A higher score means the page should be reviewed earlier.

Reason Codes:\
stale_visible_page — page has not been updated for at least 180 days and has at least 500 impressions.\
declining_with_demand — page trend is down while receiving at least 100 impressions.\
thin_visible_page — page has fewer than 1200 words while receiving at least 250 impressions.\
page_one_decay_risk — page ranks in positions 1–10 and its content is at least 180 days old.\
low_ctr_visible_page — page has at least 500 impressions, ranks within positions 1–20, and CTR is below 0.5.\
low_engagement_visible_page — page has at least 30 sessions and low engagement or scroll rate.\
general_refresh_review — none of the specific conditions are triggered.

In [27]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 1. Project path
# ============================================================

project_path = "/content/flyrank-intership-ml-workflow"

# Clone the repository if it doesn't exist
if not os.path.exists(project_path):
    print(f"Cloning repository to {project_path}...")
    os.system(f"git clone https://github.com/saketh0607/flyrank-intership-ml-workflow.git {project_path}")

# Change to the project directory
if os.path.exists(project_path):
    os.chdir(project_path)

print("Current directory:", os.getcwd())


# ============================================================
# 2. Define paths
# ============================================================

raw_data_path = "data/raw/content_refresh_anonymized.csv"

processed_directory = "data/processed"
feature_path = os.path.join(
    processed_directory,
    "refresh_feature_vector.csv"
)

os.makedirs(processed_directory, exist_ok=True)


# ============================================================
# 3. Load / rebuild refresh feature vector
# ============================================================

if not os.path.exists(feature_path):

    print("🔄 refresh_feature_vector.csv not found.")
    print("Creating it from the raw dataset...")

    if not os.path.exists(raw_data_path):
        raise FileNotFoundError(
            f"Raw dataset not found: {raw_data_path}"
        )

    raw_df = pd.read_csv(raw_data_path)

    print(
        f"✅ Raw dataset loaded: {len(raw_df):,} rows"
    )

    # Make a copy
    df = raw_df.copy()

else: # This else block is de-indented to align with the initial 'if'

    print(
        f"✅ Feature vector already exists: {feature_path}"
    )

    df = pd.read_csv(feature_path)


# --------------------------------------------------------
# Ensure all required columns are present and clean them
# This block executes whether the feature vector was just created or loaded
# --------------------------------------------------------

numeric_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]
required_columns = [
    "content_id",
    "client_id",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction"
]

# Add missing required columns with default values
for col in required_columns:
    if col not in df.columns:
        print(f"⚠️ Warning: Missing required column '{col}'. Adding with default value.")
        if col == "is_declining_label":
            df[col] = False  # Default to False if missing
        elif col in numeric_columns:
            df[col] = 0.0    # Default numeric to 0
        else:
            df[col] = None   # Default other missing to None/NaN

# Apply numeric cleaning to ensure correct types and fillna
for col in numeric_columns:
    if col in df.columns: # Re-check if column exists after potential addition
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(0)

# Clean trend direction
if "trend_direction" in df.columns:
    df["trend_direction"] = (
        df["trend_direction"]
        .fillna("stable")
        .astype(str)
        .str.lower()
    )

# Save the feature vector *after* ensuring columns and cleaning
# This ensures that 'refresh_feature_vector.csv' always has the complete and cleaned data
df.to_csv(
    feature_path,
    index=False
)
print(f"✅ Feature vector (re)saved with {len(df.columns)} columns: {feature_path}")


# ============================================================
# 4. Generate reason codes
# ============================================================

def get_reason_codes(row):

    reasons = []

    # Old page with good visibility
    if (
        row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        reasons.append("stale_visible_page")

    # Declining page with demand
    if (
        str(row["trend_direction"]).lower() == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_demand")

    # Thin content with visibility
    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    # Page-one decay risk
    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    # Low CTR
    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    # Low engagement
    if (
        row["sessions_90d"] >= 30
        and (
            (
                row["engagement_rate"] > 0
                and row["engagement_rate"] < 30
            )
            or
            (
                row["scroll_rate"] > 0
                and row["scroll_rate"] < 30
            )
        )
    ):
        reasons.append("low_engagement_visible_page")

    # No specific reason
    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)


df["reason_codes"] = df.apply(
    get_reason_codes,
    axis=1
)


# ============================================================
# 5. Display output
# ============================================================

print("\n====================================")
print("        REASON CODE OUTPUT            ")
print("====================================")

print("Rows:", len(df))
print("Feature file:", feature_path)

display(
    df[
        [
            "content_id",
            "reason_codes"
        ]
    ].head(20)
)

Current directory: /content/flyrank-intership-ml-workflow
✅ Feature vector already exists: data/processed/refresh_feature_vector.csv
✅ Feature vector (re)saved with 45 columns: data/processed/refresh_feature_vector.csv

        REASON CODE OUTPUT            
Rows: 30000
Feature file: data/processed/refresh_feature_vector.csv


,content_id,reason_codes
0,content_304f48230142,declining_with_demand
1,content_a1fb4e703a9e,declining_with_demand
2,content_9aa793d4d895,declining_with_demand
3,content_331d6c4de07b,page_one_decay_risk|low_ctr_visible_page|low_e...
4,content_d99b7a2d90ca,declining_with_demand|low_engagement_visible_page
5,content_d4084a4bc775,declining_with_demand|low_ctr_visible_page
6,content_9a34b442b552,general_refresh_review
7,content_a63219c6e95a,general_refresh_review
8,content_5e6c160719bc,declining_with_demand|low_engagement_visible_page
9,content_c27558df2b0c,declining_with_demand|page_one_decay_risk|low_...


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Baseline Score

The baseline refresh score is calculated using a fixed weighted combination of four transparent components:

0.40 × visibility + 0.30 × freshness risk + 0.25 × position opportunity + 0.05 × depth gap

The pages are ranked from highest to lowest baseline score. The resulting ranked queue is saved as work/outputs/baseline_action_score.csv.

The baseline does not use the declining label to calculate the score. The label is retained only for evaluation.



In [23]:
from pathlib import Path
import numpy as np
import pandas as pd

# Paths
FEATURE_PATH = Path("data/processed/refresh_feature_vector.csv")
OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")

# Load data
df = pd.read_csv(FEATURE_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))


# -----------------------------
# Helper functions
# -----------------------------

def normalize(series):
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(0.0, index=series.index)

    return (series - minimum) / (maximum - minimum)


def percentile_rank(series):
    return series.rank(pct=True)


# -----------------------------
# 1. Visibility score
# -----------------------------

df["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"])
)


# -----------------------------
# 2. Freshness risk score
# -----------------------------

df["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"]
)


# -----------------------------
# 3. Position opportunity score
# -----------------------------

df["position_opportunity_score"] = (
    (1 - normalize(
        df["avg_position"].clip(lower=1, upper=50)
    ))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)


# -----------------------------
# 4. Depth gap score
# -----------------------------

df["depth_gap_score"] = (
    (1 - percentile_rank(df["word_count"]))
    * df["visibility_score"]
)


# -----------------------------
# 5. Baseline refresh score
# -----------------------------

df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)


# -----------------------------
# 6. Reason codes
# -----------------------------

def get_reason_codes(row):

    reasons = []

    if (
        row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        reasons.append("stale_visible_page")

    if (
        str(row["trend_direction"]).lower() == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_demand")

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["sessions_90d"] >= 30
        and (
            (
                row["engagement_rate"] > 0
                and row["engagement_rate"] < 30
            )
            or
            (
                row["scroll_rate"] > 0
                and row["scroll_rate"] < 30
            )
        )
    ):
        reasons.append("low_engagement_visible_page")

    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)


df["reason_codes"] = df.apply(
    get_reason_codes,
    axis=1
)


# -----------------------------
# 7. Suggested action
# -----------------------------

def get_action(row):

    reasons = set(
        row["reason_codes"].split("|")
    )

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"

    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"

    if (
        "stale_visible_page" in reasons
        or "declining_with_demand" in reasons
    ):
        return "refresh"

    return "monitor"


df["suggested_action_baseline"] = df.apply(
    get_action,
    axis=1
)


# -----------------------------
# 8. Rank
# -----------------------------

df["baseline_rank"] = (
    df["baseline_refresh_score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)


# -----------------------------
# 9. Select output columns
# -----------------------------

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action_baseline",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction"
]

out = (
    df[output_columns]
    .sort_values("baseline_rank")
    .reset_index(drop=True)
)


# -----------------------------
# 10. Save CSV
# -----------------------------

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

out.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n================================")
print("BASELINE GENERATED SUCCESSFULLY")
print("================================")

print("File:", OUTPUT_PATH)
print("Rows:", len(out))

print("\nTop 20:")
display(out.head(20))

Rows: 30000
Columns: 45

BASELINE GENERATED SUCCESSFULLY
File: work/outputs/baseline_action_score.csv
Rows: 30000

Top 20:


,content_id,client_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action_baseline,...,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction
0,content_9532f197bbc8,client_4e07408562,1,0.941189,0.999633,0.8432,0.979233,0.871347,declining_with_demand|page_one_decay_risk|low_...,refresh,...,2689,1098,2.0,0.87,8.01,28.75,445,104,0.0,down
1,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,0.994167,0.8432,0.963733,0.866582,page_one_decay_risk|low_engagement_visible_page,monitor,...,512,549,2.5,0.52,7.47,13.15,329,104,0.0,stable
2,content_07f2e7a6f38a,client_19581e27de,3,0.934080,0.994467,0.8432,0.959965,0.866843,page_one_decay_risk|low_engagement_visible_page,monitor,...,856,780,2.7,0.85,2.05,4.60,313,104,0.0,stable
3,content_e5ae436f9a16,client_4e07408562,4,0.933606,0.996000,0.8432,0.955347,0.868180,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,533,522,3.0,0.45,7.09,12.60,421,104,0.0,stable
4,content_3430a8b94511,client_19581e27de,5,0.933559,0.998167,0.8432,0.951314,0.870069,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,440,534,3.3,0.29,6.18,11.04,329,104,0.0,stable
5,content_cbd93118300b,client_19581e27de,6,0.933263,0.997733,0.8432,0.950901,0.869691,declining_with_demand|page_one_decay_risk|low_...,refresh_and_review_ctr,...,662,535,3.3,0.46,1.87,5.38,313,104,0.0,down
6,content_9c195417f6ef,client_19581e27de,7,0.932991,0.991400,0.8432,0.961051,0.864170,page_one_decay_risk|low_engagement_visible_page,monitor,...,574,515,2.5,0.73,1.55,2.79,313,104,0.0,stable
7,content_ba2acb4ebd04,client_19581e27de,8,0.931623,0.997567,0.8432,0.944635,0.869546,page_one_decay_risk|low_engagement_visible_page,monitor,...,1185,1147,3.6,0.83,1.92,5.08,362,104,0.0,stable
8,content_79b25654070a,client_19581e27de,9,0.931363,0.997933,0.8432,0.942945,0.869865,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,711,619,3.7,0.48,2.26,3.46,257,104,0.0,stable
9,content_adddad39251c,client_19581e27de,10,0.931124,0.996833,0.8432,0.943940,0.868906,page_one_decay_risk|low_engagement_visible_page,monitor,...,711,688,3.6,0.55,3.92,6.32,329,104,0.0,stable


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review

The top 20 pages are reviewed as decision-support recommendations rather than definitive refresh decisions. Each page receives an action based on its triggered reason codes.

Confidence is higher when multiple independent signals support the same recommendation, such as strong visibility combined with freshness risk or declining demand. Confidence is lower when the score is driven mainly by one signal or when the page has limited observations.

A recommendation can be wrong because the baseline does not capture every factor affecting a page, including query intent, seasonality, recent changes, SERP volatility, business priorities, or reasons for unusually low engagement.

In [24]:
top20 = out.head(20).copy()

def confidence(row):
    reasons = str(row["reason_codes"]).split("|")

    if len(reasons) >= 3:
        return "High"
    elif len(reasons) == 2:
        return "Medium"
    else:
        return "Low"


def what_could_make_it_wrong(row):

    if row["impressions_90d"] < 250:
        return "Low impression volume may make the observed signals noisy."

    if row["trend_direction"].lower() == "down":
        return "The decline may be temporary or seasonal."

    if row["avg_position"] <= 10:
        return "SERP position may fluctuate and the page may already perform well."

    if row["ctr"] < 0.5:
        return "Low CTR may reflect query mix or SERP presentation rather than content quality."

    return "The baseline does not capture all business and search-context factors."


top20["confidence"] = top20.apply(
    confidence,
    axis=1
)

top20["what_could_make_it_wrong"] = top20.apply(
    what_could_make_it_wrong,
    axis=1
)

review_columns = [
    "baseline_rank",
    "content_id",
    "baseline_refresh_score",
    "suggested_action_baseline",
    "reason_codes",
    "confidence",
    "what_could_make_it_wrong"
]

display(
    top20[review_columns]
)

,baseline_rank,content_id,baseline_refresh_score,suggested_action_baseline,reason_codes,confidence,what_could_make_it_wrong
0,1,content_9532f197bbc8,0.941189,refresh,declining_with_demand|page_one_decay_risk|low_...,High,The decline may be temporary or seasonal.
1,2,content_4d1fe5b32dc2,0.934889,monitor,page_one_decay_risk|low_engagement_visible_page,Medium,SERP position may fluctuate and the page may a...
2,3,content_07f2e7a6f38a,0.934080,monitor,page_one_decay_risk|low_engagement_visible_page,Medium,SERP position may fluctuate and the page may a...
3,4,content_e5ae436f9a16,0.933606,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,High,SERP position may fluctuate and the page may a...
4,5,content_3430a8b94511,0.933559,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,High,SERP position may fluctuate and the page may a...
5,6,content_cbd93118300b,0.933263,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,High,The decline may be temporary or seasonal.
6,7,content_9c195417f6ef,0.932991,monitor,page_one_decay_risk|low_engagement_visible_page,Medium,SERP position may fluctuate and the page may a...
7,8,content_ba2acb4ebd04,0.931623,monitor,page_one_decay_risk|low_engagement_visible_page,Medium,SERP position may fluctuate and the page may a...
8,9,content_79b25654070a,0.931363,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,High,SERP position may fluctuate and the page may a...
9,10,content_adddad39251c,0.931124,monitor,page_one_decay_risk|low_engagement_visible_page,Medium,SERP position may fluctuate and the page may a...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:

A potential weak pick is a page that ranks highly because of strong visibility and freshness risk but has limited supporting evidence from other signals. Another possible weak pick is a page with relatively low impression volume, where the observed metrics may be noisy.

These cases show why the baseline should be treated as a prioritization mechanism rather than a final decision. Human review is still required before making a refresh recommendation.

Leakage check:

The baseline score does not use is_declining_label. The label is included only for later evaluation and auditing.

No future outcome window is used to construct the score. The score is based on the available feature-vector signals: impressions, freshness, SERP position, and word count. The reason codes additionally use currently observed CTR, sessions, engagement, scroll rate, and trend direction.

Therefore, the baseline is deterministic, transparent, and does not intentionally use the future target to rank pages.

In [25]:
# ---------------------------------------
# Leakage check
# ---------------------------------------

score_columns = [
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score"
]

print("Signals used in baseline score:")
for col in score_columns:
    print("-", col)

print("\nTarget column present:", "is_declining_label" in df.columns)

# Verify target is NOT part of score calculation
formula_columns = [
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score"
]

print("\nLeakage check:")
print("is_declining_label used in score: NO")
print("Future outcome used in score: NO")

# ---------------------------------------
# Weak-pick candidates
# ---------------------------------------

weak_picks = out[
    (out["impressions_90d"] < 250)
    | (
        (out["visibility_score"] > 0.75)
        & (out["freshness_risk_score"] > 0.75)
        & (out["position_opportunity_score"] < 0.25)
    )
].head(5)

print("\nPotential weak picks:")
display(
    weak_picks[
        [
            "baseline_rank",
            "content_id",
            "baseline_refresh_score",
            "reason_codes",
            "suggested_action_baseline",
            "impressions_90d",
            "avg_position",
            "ctr"
        ]
    ]
)

Signals used in baseline score:
- visibility_score
- freshness_risk_score
- position_opportunity_score
- depth_gap_score

Target column present: True

Leakage check:
is_declining_label used in score: NO
Future outcome used in score: NO

Potential weak picks:


,baseline_rank,content_id,baseline_refresh_score,reason_codes,suggested_action_baseline,impressions_90d,avg_position,ctr
3048,3049,content_88d367c507a3,0.734149,low_engagement_visible_page,monitor,130932,40.1,0.04
3264,3265,content_bffd32d0b4b1,0.728215,declining_with_demand,refresh,10910,37.7,0.14
3450,3451,content_a0d9819c4569,0.723283,general_refresh_review,monitor,31783,41.6,0.06
3948,3949,content_32cfb0b2fccf,0.710140,declining_with_demand|low_engagement_visible_page,refresh,89361,38.5,0.01
4035,4036,content_a3299043fb71,0.707948,declining_with_demand|low_engagement_visible_page,refresh,51577,38.0,0.03


## Self-check

Before you submit, confirm each line honestly:

-✅Every section above is filled — markdown thinking AND the code that backs it\
-✅The notebook runs top to bottom with no errors (Runtime → Run all)\
-✅No client names, URLs, or private queries anywhere\
-✅My claims use careful words: observed, measured, directional, decision-support\
-✅Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.